# DSA4213 Assignment 1 — Colab workflow

Clean, reproducible workflow for the quick check, the 8-epoch supplementary run, and the four 20-epoch runs (seeds 42, 1, 2, and 3). Run cells from top to bottom in a Colab GPU runtime.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Upload and extract the project

Upload `assignment1.zip`. The cell copies it to Drive and extracts it under `/content/drive/MyDrive/dsa4213/`.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
import zipfile

uploaded = files.upload()
if "assignment1.zip" not in uploaded:
    raise FileNotFoundError("Please upload a file named assignment1.zip")

drive_root = Path("/content/drive/MyDrive/dsa4213")
drive_root.mkdir(parents=True, exist_ok=True)
archive = drive_root / "assignment1.zip"
archive.write_bytes(uploaded["assignment1.zip"])
with zipfile.ZipFile(archive) as zf:
    zf.extractall(drive_root)

project_root = drive_root / "assignment1"
if not (project_root / "config.yaml").exists():
    raise FileNotFoundError(f"Expected project root not found: {project_root}")
print(project_root)

## Check the GPU and install dependencies

Use Colab's preinstalled PyTorch stack; install only the missing packages as binary wheels.

In [ ]:
%cd /content/drive/MyDrive/dsa4213/assignment1
!nvidia-smi
!pip install -q --only-binary=:all: gensim datasets

## Quick pipeline check

This writes smoke-test artifacts only under `results/*/quick/` and `checkpoints/*/quick/`.

In [ ]:
%cd /content/drive/MyDrive/dsa4213/assignment1
!python run_all.py --quick

## Run the 8-epoch seed-42 supplement

Parse YAML and modify only `train.epochs`; `word2vec.epochs` remains 5.

In [ ]:
%cd /content/drive/MyDrive/dsa4213/assignment1
from pathlib import Path
import yaml

config_path = Path("config.yaml")
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
config["seed"] = 42
config["train"]["epochs"] = 8
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
assert config["train"]["epochs"] == 8
assert config["word2vec"]["epochs"] == 5
!python run_all.py

## Preserve the 8-epoch run

Rename both output directories before starting the independent 20-epoch run.

In [ ]:
%cd /content/drive/MyDrive/dsa4213/assignment1
from pathlib import Path

for source_name, target_name in (("results", "results_e8"), ("checkpoints", "checkpoints_e8")):
    source, target = Path(source_name), Path(target_name)
    if not source.exists():
        raise FileNotFoundError(source)
    if target.exists():
        raise FileExistsError(f"Refusing to overwrite {target}")
    source.rename(target)

## Run the 20-epoch seed-42 experiment

Restore the main training budget while retaining the Word2Vec budget of five epochs.

In [ ]:
%cd /content/drive/MyDrive/dsa4213/assignment1
from pathlib import Path
import yaml

config_path = Path("config.yaml")
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
config["seed"] = 42
config["train"]["epochs"] = 20
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
assert config["train"]["epochs"] == 20
assert config["word2vec"]["epochs"] == 5
!python run_all.py

## Run seeds 1, 2, and 3

`run_seeds.sh` first stages the completed seed-42 run, executes the three additional seeds, and restores seed 42 in `config.yaml`.

In [ ]:
%cd /content/drive/MyDrive/dsa4213/assignment1
!bash run_seeds.sh

## Normalize the main-run name and aggregate seeds

Rename the staged seed-42 results to the submitted `results_e20` layout, then compute four-seed summaries and consistency checks.

In [ ]:
%cd /content/drive/MyDrive/dsa4213/assignment1
from pathlib import Path

source, target = Path("results_seed42"), Path("results_e20")
if not source.exists():
    raise FileNotFoundError(source)
if target.exists():
    raise FileExistsError(f"Refusing to overwrite {target}")
source.rename(target)
!python aggregate_seeds.py

## Archive and download formal results

Create one archive containing the 8-epoch supplement, all four 20-epoch result folders, and the generated analysis summaries.

In [ ]:
%cd /content/drive/MyDrive/dsa4213/assignment1
from google.colab import files
from pathlib import Path
import zipfile

folders = ["results_e8", "results_e20", "results_seed1", "results_seed2", "results_seed3", "analysis"]
missing = [name for name in folders if not Path(name).exists()]
if missing:
    raise FileNotFoundError(f"Missing folders: {missing}")
archive = Path("final_results.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in folders:
        for path in Path(folder).rglob("*"):
            if path.is_file():
                zf.write(path, path.as_posix())
files.download(str(archive))